# Predictive Modeling: a beginner's first machine-learning project

Hi! In this notebook, we'll teach a few simple models to make predictions, then check how they did on examples they haven't seen before. You can run each cell with **Shift + Enter**. Read the short notes as you go; the goal is to understand the process, not just get a score.

**You'll practice:** classification (choosing a label), regression (predicting a number), train/test splits, and a few ways to evaluate predictions. No data downloads are needed.

## 1. Get our tools ready

These libraries do different jobs: pandas keeps results in tidy tables, matplotlib draws charts, and scikit-learn gives us practice data and machine-learning tools. If this cell says a package is missing, follow the setup steps in `README.md`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, auc, confusion_matrix,
    mean_squared_error, precision_score, recall_score, r2_score, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_SEED = 42
TEST_SIZE = 0.20  # Keep 20% of examples aside for our final check.
print('Tools are ready. Let’s make some predictions!')

## 2. First task: predict a label

Our first practice data describes measurements from cell samples. Each row is one sample (an **example**); the columns are its **features**; the answer we're trying to predict is its **target**. Here the target has two labels: malignant and benign. This is called **classification**.

> This is a teaching exercise using a small built-in dataset. It is not a diagnostic tool or medical advice.

In [ ]:
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target  # In this dataset: 0 = malignant, 1 = benign.

print(f'Number of examples: {len(X)}')
print(f'Number of features per example: {X.shape[1]}')
print('A few feature names:', list(X.columns[:5]))
print('Target label counts:')
print(pd.Series(y).map({0: 'malignant', 1: 'benign'}).value_counts())
X.head()  # Peek at the first five examples.

## 3. Save some examples for an honest check

We'll let the models learn from 80% of the examples, and hold 20% back. The held-back **test set** acts like new examples the model hasn't seen during learning. `stratify=y` keeps roughly the same mix of labels in both groups. The seed makes the split repeatable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Training examples: {len(X_train)}')
print(f'Test examples: {len(X_test)}')

## 4. Give three models a try

Logistic Regression draws a relatively simple boundary between labels. A Decision Tree asks a chain of yes/no questions. A Random Forest combines many trees, which can make its predictions steadier. These are different ways of finding patterns; we'll compare them on the same held-out examples.

We scale the measurements for Logistic Regression because features with very different number ranges can make learning harder. The tree models don't need that step.

In [ ]:
classifiers = {
    'Logistic Regression': make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=5000, random_state=RANDOM_SEED)
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=RANDOM_SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
    ),
}

class_predictions = {}
classification_rows = []
for name, model in classifiers.items():
    model.fit(X_train, y_train)  # Learn patterns from the training examples.
    predicted = model.predict(X_test)
    probability_benign = model.predict_proba(X_test)[:, 1]
    class_predictions[name] = (predicted, probability_benign)
    classification_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predicted),
        'Benign precision': precision_score(y_test, predicted, pos_label=1),
        'Benign recall': recall_score(y_test, predicted, pos_label=1),
        'Benign ROC AUC': auc(*roc_curve(y_test, probability_benign)[:2]),
    })

classification_results = pd.DataFrame(classification_rows).sort_values('Accuracy', ascending=False)
classification_results.round(3)

### How should I read those scores?

- **Accuracy**: what fraction of all test predictions were right?
- **Precision (benign)**: when the model said benign, how often was that right?
- **Recall (benign)**: of the examples that really were benign, how many did it find?
- **ROC AUC**: how well does the model rank benign examples above malignant ones across many possible cutoffs? A score near 1 is strong ranking on this test set; around 0.5 is similar to random ordering.

No one score tells the whole story. The 'positive' class for precision, recall, and ROC here is benign (label 1); the printed confusion matrices show both classes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Classification: predictions on examples held back from training', fontsize=15)

for ax, (name, (predicted, probability)) in zip(axes.flat[:3], class_predictions.items()):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, predicted), display_labels=cancer.target_names
    ).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(name)
    ax.tick_params(axis='x', labelrotation=15)

ax = axes.flat[3]
for name, (_, probability) in class_predictions.items():
    fpr, tpr, _ = roc_curve(y_test, probability)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Random baseline')
ax.set(title='ROC curves (positive class: benign)', xlabel='False positive rate', ylabel='True positive rate')
ax.legend(fontsize=8, loc='lower right')
fig.tight_layout()
plt.show()

### A quick guide to the charts

In each confusion matrix, rows are the true labels and columns are the model's guesses. Numbers on the diagonal are correct guesses; numbers off the diagonal are mix-ups. In the ROC chart, curves nearer the top-left generally indicate better ranking. These results describe only this held-out sample, not how a medical tool would perform in practice.

## 5. A different kind of question: predict a number

So far, models chose between labels. Now we'll predict a number: a measure of disease progression in another built-in teaching dataset. Predicting a numeric value is called **regression**. This distinction matters: confusion matrices and ROC curves are for classification, while R² and RMSE are for regression.

Linear Regression tries to fit a straight-line relationship. A Decision Tree splits examples into groups, and a Random Forest averages many trees.

In [ ]:
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = diabetes.target

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

regressors = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, min_samples_leaf=3, random_state=RANDOM_SEED, n_jobs=-1
    ),
}
reg_predictions = {}
regression_rows = []
for name, model in regressors.items():
    model.fit(X_reg_train, y_reg_train)
    predicted = model.predict(X_reg_test)
    reg_predictions[name] = predicted
    regression_rows.append({
        'Model': name,
        'R²': r2_score(y_reg_test, predicted),
        'RMSE': mean_squared_error(y_reg_test, predicted) ** 0.5,
    })

regression_results = pd.DataFrame(regression_rows).sort_values('R²', ascending=False)
regression_results.round(3)

### Regression scores in plain language

**R²** is a rough measure of how much variation in the test answers is accounted for by the predictions; closer to 1 is better, and it can even be negative. **RMSE** is an average-sized prediction error, expressed in the same units as the target; smaller is better. Don't compare this RMSE to the classification scores: they answer different questions.

In [ ]:
best_name = regression_results.iloc[0]['Model']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_reg_test, reg_predictions[best_name], alpha=0.75)
low, high = float(y_reg_test.min()), float(y_reg_test.max())
axes[0].plot([low, high], [low, high], '--', color='firebrick')
axes[0].set(title=f'Actual vs. predicted ({best_name})', xlabel='Actual target', ylabel='Predicted target')
axes[1].bar(regression_results['Model'], regression_results['R²'])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(title='R² by model (higher is better)', ylabel='R²')
axes[1].tick_params(axis='x', labelrotation=18)
fig.tight_layout()
plt.show()

## 6. What did we learn?

We kept test examples separate while models learned, tried a few different learning methods, and chose measurements that match the question: labels or numbers. The best score in this exercise doesn't mean one model is always best. Results can change with different data, splits, and choices about what kinds of mistakes matter.

### Your turn

1. Which classification model had the highest accuracy? Did it also have the highest ROC AUC?
2. Look at the confusion matrix with the fewest off-diagonal errors. Which kinds of mistakes remain?
3. Which regression model had the lowest RMSE? What does that error mean in the target's units?
4. Try changing `max_depth` on the Decision Tree. What changed in its scores?

**Takeaway:** a useful model is more than a high score. Understand the data, keep a fair test set, and choose measures that fit the real question.